# Module 5: Multi-Objective Feature Selection with NSGA-II

## Learning Objectives

By completing this notebook, you will:
1. Understand multi-objective optimization and Pareto optimality
2. Implement NSGA-II for feature selection with DEAP
3. Optimize multiple conflicting objectives simultaneously
4. Visualize and interpret Pareto fronts
5. Select solutions from Pareto front based on preferences
6. Apply NSGA-II to real-world feature selection problems

## Prerequisites

- Module 4 completed (GA implementation)
- Understanding of Pareto optimality
- DEAP library proficiency
- Familiarity with dominated solutions

## Estimated Time: 90 minutes

---

In [ ]:
learning_objectives(["Understand multi-objective optimization and Pareto optimality", "Implement NSGA-II for feature selection with DEAP", "Optimize multiple conflicting objectives simultaneously", "Visualize and interpret Pareto fronts", "Select solutions from Pareto front based on preferences", "Apply NSGA-II to real-world feature selection problems"])

In [ ]:
section_divider("Multi-Objective Optimization Fundamentals")

## 1. Multi-Objective Optimization Fundamentals

### Key Concept: Real-world problems have multiple conflicting objectives.

**Why Multi-Objective for Feature Selection?**

Single-objective GA optimizes:
- `fitness = accuracy - parsimony_penalty`

Problems:
1. **Weight tuning**: Choosing parsimony penalty weight is arbitrary
2. **Trade-off hidden**: Single fitness obscures accuracy-complexity trade-off
3. **User preferences**: Different users want different trade-offs

**Multi-Objective Approach:**
- Objective 1: Maximize accuracy
- Objective 2: Minimize number of features
- Result: Pareto front of non-dominated solutions

**Pareto Dominance:**
- Solution A dominates B if A is better in all objectives
- Non-dominated solutions form the Pareto front
- No solution on front dominates any other

**NSGA-II Algorithm:**
- Non-dominated Sorting Genetic Algorithm II
- Fast non-dominated sorting
- Crowding distance for diversity
- Elitist selection

In [ ]:
callout("Real-world problems have multiple conflicting objectives.", kind="insight")

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# DEAP imports
from deap import base, creator, tools, algorithms
import random

# Set random seeds
np.random.seed(42)
random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")

In [ ]:
apply_course_theme()
apply_plot_theme()

### 1.1 Load and Prepare Dataset

In [ ]:
# Purpose: Prepare dataset for multi-objective feature selection
# Key Concept: Dataset with clear accuracy-complexity trade-off

# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

N_FEATURES = X_train_scaled.shape[1]

print(f"Dataset: {X_train_scaled.shape[0]} train, {X_test_scaled.shape[0]} test")
print(f"Features: {N_FEATURES}")
print(f"\nFeatures: {X.columns.tolist()}")

In [ ]:
section_divider("NSGA-II Implementation")

## 2. NSGA-II Implementation

### Key Concept: Multi-objective fitness returns tuple with multiple values.

In [ ]:
callout("Multi-objective fitness returns tuple with multiple values.", kind="insight")

### 2.1 Setup Multi-Objective DEAP

In [ ]:
# Purpose: Create multi-objective fitness and individual types
# Key Concept: Fitness weights define optimization direction

# Clean up existing types
if hasattr(creator, "FitnessMulti"):
    del creator.FitnessMulti
if hasattr(creator, "Individual"):
    del creator.Individual

# Create multi-objective fitness
# Objective 1: Maximize accuracy (weight = 1.0)
# Objective 2: Minimize number of features (weight = -1.0)
creator.create("FitnessMulti", base.Fitness, weights=(1.0, -1.0))

# Create individual
creator.create("Individual", list, fitness=creator.FitnessMulti)

print("Multi-objective DEAP types created:")
print(f"  Fitness weights: {creator.FitnessMulti.weights}")
print(f"    1.0  = Maximize accuracy")
print(f"    -1.0 = Minimize features (negative weight)")

### 2.2 Multi-Objective Fitness Function

In [ ]:
# Purpose: Define fitness function returning multiple objectives
# Key Concept: Return tuple with (accuracy, n_features)

def evaluate_multi_objective(individual, X_data, y_data):
    """
    Evaluate two objectives: accuracy and number of features.
    
    Parameters
    ----------
    individual : list
        Binary chromosome
    X_data : pd.DataFrame
        Training features
    y_data : np.ndarray
        Training labels
    
    Returns
    -------
    objectives : tuple
        (accuracy, n_features)
    """
    n_selected = sum(individual)
    
    # Constraint: at least one feature
    if n_selected == 0:
        return (0.0, N_FEATURES)  # Worst possible values
    
    # Select features
    feature_mask = np.array(individual, dtype=bool)
    X_selected = X_data.iloc[:, feature_mask]
    
    # Evaluate accuracy with cross-validation
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X_selected, y_data, cv=5, scoring='accuracy')
    accuracy = scores.mean()
    
    # Objective 1: Maximize accuracy
    obj1 = accuracy
    
    # Objective 2: Minimize number of features
    # Note: DEAP handles minimization via negative weight
    obj2 = n_selected
    
    return (obj1, obj2)

# Test fitness function
test_individual = [1] * 10 + [0] * (N_FEATURES - 10)
test_fitness = evaluate_multi_objective(test_individual, X_train_scaled, y_train)

print(f"Test individual fitness:")
print(f"  Accuracy: {test_fitness[0]:.4f}")
print(f"  Features: {test_fitness[1]}")
print(f"\nFitness tuple: {test_fitness}")

### 2.3 Setup NSGA-II Toolbox

In [ ]:
# Purpose: Register NSGA-II operators
# Key Concept: Use selNSGA2 for selection

# Create toolbox
toolbox = base.Toolbox()

# Register components
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=N_FEATURES
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_multi_objective, X_data=X_train_scaled, y_data=y_train)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selNSGA2)  # NSGA-II selection

print("NSGA-II toolbox configured:")
print(f"  Individual length: {N_FEATURES}")
print(f"  Selection: NSGA-II")
print(f"  Crossover: Two-point")
print(f"  Mutation: Flip bit (indpb=0.05)")

### 2.4 Run NSGA-II

In [ ]:
# Purpose: Execute NSGA-II algorithm
# Key Concept: Generate Pareto front of non-dominated solutions

# Statistics for multi-objective
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", lambda fits: np.mean(fits, axis=0))
stats.register("min", lambda fits: np.min(fits, axis=0))
stats.register("max", lambda fits: np.max(fits, axis=0))

# Parameters
POPULATION_SIZE = 100  # Larger for better Pareto front coverage
MAX_GENERATIONS = 50
P_CROSSOVER = 0.7
P_MUTATION = 0.2

print("Running NSGA-II...")
print(f"  Population: {POPULATION_SIZE}")
print(f"  Generations: {MAX_GENERATIONS}")
print(f"\nThis may take a few minutes...\n")

# Create initial population
population = toolbox.population(n=POPULATION_SIZE)

# Evaluate initial population
fitnesses = map(toolbox.evaluate, population)
for ind, fit in zip(population, fitnesses):
    ind.fitness.values = fit

# Evolution loop
for gen in range(MAX_GENERATIONS):
    # Select next generation
    offspring = toolbox.select(population, len(population))
    offspring = [toolbox.clone(ind) for ind in offspring]
    
    # Apply crossover
    for i in range(1, len(offspring), 2):
        if random.random() < P_CROSSOVER:
            offspring[i-1], offspring[i] = toolbox.mate(offspring[i-1], offspring[i])
            del offspring[i-1].fitness.values
            del offspring[i].fitness.values
    
    # Apply mutation
    for mutant in offspring:
        if random.random() < P_MUTATION:
            toolbox.mutate(mutant)
            del mutant.fitness.values
    
    # Evaluate individuals with invalid fitness
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit
    
    # Replace population
    population = offspring
    
    # Progress
    if (gen + 1) % 10 == 0:
        print(f"Generation {gen + 1}/{MAX_GENERATIONS}")

print("\nNSGA-II completed!")

### 2.5 Extract Pareto Front

In [ ]:
# Purpose: Extract non-dominated solutions
# Key Concept: Pareto front contains all optimal trade-offs

# Get Pareto front (first front only)
pareto_front = tools.sortNondominated(population, len(population), first_front_only=True)[0]

print(f"Pareto Front Statistics:")
print("="*70)
print(f"Solutions on Pareto front: {len(pareto_front)}")
print(f"\nTop 10 solutions (sorted by accuracy):")
print(f"{'Rank':>4} {'Accuracy':>10} {'Features':>10}")
print("-"*26)

# Sort by accuracy (descending)
pareto_sorted = sorted(pareto_front, key=lambda ind: ind.fitness.values[0], reverse=True)

for i, ind in enumerate(pareto_sorted[:10], 1):
    accuracy = ind.fitness.values[0]
    n_features = int(ind.fitness.values[1])
    print(f"{i:4d} {accuracy:10.4f} {n_features:10d}")

# Statistics
accuracies = [ind.fitness.values[0] for ind in pareto_front]
n_features_list = [int(ind.fitness.values[1]) for ind in pareto_front]

print(f"\nAccuracy range: [{min(accuracies):.4f}, {max(accuracies):.4f}]")
print(f"Features range: [{min(n_features_list)}, {max(n_features_list)}]")

In [ ]:
section_divider("Pareto Front Visualization and Analysis")

## 3. Pareto Front Visualization and Analysis

### Key Concept: Visualize trade-offs between objectives.

In [ ]:
callout("Visualize trade-offs between objectives.", kind="insight")

In [ ]:
# Purpose: Visualize Pareto front and trade-offs
# Key Concept: Each point represents an optimal trade-off

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Extract objectives for all population
all_accuracies = [ind.fitness.values[0] for ind in population]
all_n_features = [int(ind.fitness.values[1]) for ind in population]

# Extract Pareto front objectives
pareto_accuracies = [ind.fitness.values[0] for ind in pareto_front]
pareto_n_features = [int(ind.fitness.values[1]) for ind in pareto_front]

# Plot 1: Pareto front vs all solutions
axes[0].scatter(all_n_features, all_accuracies, alpha=0.3, s=30,
               color='lightblue', edgecolor='black', linewidth=0.5,
               label='Dominated solutions')
axes[0].scatter(pareto_n_features, pareto_accuracies, alpha=0.9, s=100,
               color='red', marker='*', edgecolor='black', linewidth=1,
               label='Pareto front', zorder=5)

axes[0].set_xlabel('Number of Features', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('NSGA-II: Pareto Front', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Plot 2: Pareto front with annotations
axes[1].plot(pareto_n_features, pareto_accuracies, 'r*-', 
            markersize=12, linewidth=2, alpha=0.7, label='Pareto front')

# Annotate interesting points
# Highest accuracy
max_acc_idx = np.argmax(pareto_accuracies)
axes[1].annotate('Highest accuracy',
                xy=(pareto_n_features[max_acc_idx], pareto_accuracies[max_acc_idx]),
                xytext=(10, -20), textcoords='offset points',
                fontsize=10, color='darkgreen',
                arrowprops=dict(arrowstyle='->', color='darkgreen'))

# Fewest features
min_feat_idx = np.argmin(pareto_n_features)
axes[1].annotate('Fewest features',
                xy=(pareto_n_features[min_feat_idx], pareto_accuracies[min_feat_idx]),
                xytext=(10, 20), textcoords='offset points',
                fontsize=10, color='darkblue',
                arrowprops=dict(arrowstyle='->', color='darkblue'))

axes[1].set_xlabel('Number of Features', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Pareto Front: Trade-off Curve', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Red stars show Pareto-optimal solutions")
print("  - Moving right increases features (complexity)")
print("  - Moving up increases accuracy")
print("  - No solution dominates another on the front")
print("  - Choose based on your preference (accuracy vs simplicity)")

In [ ]:
section_divider("Solution Selection from Pareto Front")

## 4. Solution Selection from Pareto Front

### Key Concept: Choose solution based on user preferences or constraints.

### 4.1 Knee Point Selection

In [ ]:
# Purpose: Find knee point (best compromise)
# Key Concept: Maximum distance from line connecting extremes

def find_knee_point(pareto_front):
    """
    Find knee point on Pareto front.
    
    Knee point represents best balance between objectives.
    
    Parameters
    ----------
    pareto_front : list
        Pareto-optimal individuals
    
    Returns
    -------
    knee_solution : Individual
        Solution at knee point
    """
    # Sort by first objective (accuracy)
    sorted_front = sorted(pareto_front, key=lambda ind: ind.fitness.values[1])
    
    # Extract objectives
    objectives = np.array([ind.fitness.values for ind in sorted_front])
    
    # Normalize to [0, 1]
    normalized = (objectives - objectives.min(axis=0)) / \
                 (objectives.max(axis=0) - objectives.min(axis=0) + 1e-10)
    
    # Line from first to last point
    p1 = normalized[0]
    p2 = normalized[-1]
    
    # Calculate distance from each point to line
    distances = []
    for point in normalized:
        # Distance from point to line
        d = np.abs(np.cross(p2 - p1, point - p1)) / np.linalg.norm(p2 - p1)
        distances.append(d)
    
    # Knee point is maximum distance
    knee_idx = np.argmax(distances)
    knee_solution = sorted_front[knee_idx]
    
    return knee_solution, knee_idx

# Find knee point
knee_solution, knee_idx = find_knee_point(pareto_front)
knee_accuracy = knee_solution.fitness.values[0]
knee_n_features = int(knee_solution.fitness.values[1])

print("Knee Point Solution:")
print("="*70)
print(f"Accuracy: {knee_accuracy:.4f}")
print(f"Features: {knee_n_features}")
print(f"\nThis represents the best balance between accuracy and complexity.")

# Visualize knee point
plt.figure(figsize=(10, 6))
plt.plot(pareto_n_features, pareto_accuracies, 'r*-', 
        markersize=10, linewidth=2, alpha=0.5, label='Pareto front')
plt.scatter([knee_n_features], [knee_accuracy], s=200, 
           color='green', marker='D', edgecolor='black', linewidth=2,
           label='Knee point', zorder=5)
plt.xlabel('Number of Features', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Knee Point on Pareto Front', fontsize=13)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Constraint-Based Selection

In [ ]:
# Purpose: Select solution based on constraints
# Key Concept: Filter Pareto front by requirements

def select_by_constraint(pareto_front, min_accuracy=None, max_features=None):
    """
    Select solution from Pareto front based on constraints.
    
    Parameters
    ----------
    pareto_front : list
        Pareto-optimal individuals
    min_accuracy : float, optional
        Minimum required accuracy
    max_features : int, optional
        Maximum allowed features
    
    Returns
    -------
    best_solution : Individual
        Best solution satisfying constraints
    """
    candidates = pareto_front
    
    # Filter by accuracy
    if min_accuracy is not None:
        candidates = [ind for ind in candidates if ind.fitness.values[0] >= min_accuracy]
    
    # Filter by features
    if max_features is not None:
        candidates = [ind for ind in candidates if ind.fitness.values[1] <= max_features]
    
    if not candidates:
        print("Warning: No solution satisfies constraints!")
        return None
    
    # Among candidates, prefer fewest features
    best_solution = min(candidates, key=lambda ind: ind.fitness.values[1])
    
    return best_solution

# Example 1: Accuracy must be >= 0.95
solution_acc = select_by_constraint(pareto_front, min_accuracy=0.95)
if solution_acc:
    print("Solution with accuracy >= 0.95:")
    print(f"  Accuracy: {solution_acc.fitness.values[0]:.4f}")
    print(f"  Features: {int(solution_acc.fitness.values[1])}")

# Example 2: Features must be <= 10
solution_feat = select_by_constraint(pareto_front, max_features=10)
if solution_feat:
    print(f"\nSolution with features <= 10:")
    print(f"  Accuracy: {solution_feat.fitness.values[0]:.4f}")
    print(f"  Features: {int(solution_feat.fitness.values[1])}")

# Example 3: Both constraints
solution_both = select_by_constraint(pareto_front, min_accuracy=0.93, max_features=15)
if solution_both:
    print(f"\nSolution with accuracy >= 0.93 AND features <= 15:")
    print(f"  Accuracy: {solution_both.fitness.values[0]:.4f}")
    print(f"  Features: {int(solution_both.fitness.values[1])}")

In [ ]:
section_divider("Three-Objective Feature Selection")

## 5. Three-Objective Feature Selection

### Key Concept: Optimize accuracy, features, and interpretability simultaneously.

In [ ]:
# Purpose: Extend to three objectives
# Key Concept: Add diversity/correlation as third objective

# Clean up
if hasattr(creator, "FitnessTriple"):
    del creator.FitnessTriple
if hasattr(creator, "IndividualTriple"):
    del creator.IndividualTriple

# Create three-objective fitness
# Obj 1: Maximize accuracy
# Obj 2: Minimize features
# Obj 3: Minimize average correlation (promote diversity)
creator.create("FitnessTriple", base.Fitness, weights=(1.0, -1.0, -1.0))
creator.create("IndividualTriple", list, fitness=creator.FitnessTriple)

def evaluate_three_objectives(individual, X_data, y_data):
    """
    Evaluate three objectives.
    
    Returns
    -------
    objectives : tuple
        (accuracy, n_features, avg_correlation)
    """
    n_selected = sum(individual)
    
    if n_selected == 0:
        return (0.0, N_FEATURES, 1.0)
    
    # Select features
    feature_mask = np.array(individual, dtype=bool)
    X_selected = X_data.iloc[:, feature_mask]
    
    # Objective 1: Accuracy
    model = LogisticRegression(max_iter=1000, random_state=42)
    scores = cross_val_score(model, X_selected, y_data, cv=3, scoring='accuracy')
    accuracy = scores.mean()
    
    # Objective 2: Number of features
    n_features = n_selected
    
    # Objective 3: Average absolute correlation (lower = more diverse)
    if n_selected > 1:
        corr_matrix = X_selected.corr().abs()
        # Get upper triangle (exclude diagonal)
        upper_triangle = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        avg_correlation = upper_triangle.stack().mean()
    else:
        avg_correlation = 0.0
    
    return (accuracy, n_features, avg_correlation)

# Test three-objective fitness
test_fitness_3obj = evaluate_three_objectives(test_individual, X_train_scaled, y_train)
print(f"Three-objective fitness test:")
print(f"  Accuracy: {test_fitness_3obj[0]:.4f}")
print(f"  Features: {test_fitness_3obj[1]}")
print(f"  Avg Correlation: {test_fitness_3obj[2]:.4f}")
print(f"\nLower correlation means more diverse (less redundant) features.")

In [ ]:
section_divider("Exercises")

## 6. Exercises

### Exercise 6.1: Run Three-Objective NSGA-II

**Task**: Complete the three-objective optimization, run NSGA-II, and visualize the 3D Pareto front.

**Expected Output**: 3D scatter plot showing Pareto front in (accuracy, features, correlation) space.

<details>
<summary>Hint 1</summary>
Create new toolbox with FitnessTriple and evaluate_three_objectives.
</details>

<details>
<summary>Hint 2</summary>
Use `ax = plt.axes(projection='3d')` for 3D visualization.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

### Exercise 6.2: Preference Articulation

**Task**: Implement weighted Tchebycheff scalarization to find solution closest to user-specified reference point (e.g., accuracy=0.98, features=5).

**Expected Output**: Solution closest to desired trade-off.

<details>
<summary>Hint</summary>
Normalize objectives, then calculate: `max(weights[i] * |objective[i] - reference[i]|)` for each solution.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

### Exercise 6.3: Hypervolume Indicator

**Task**: Calculate hypervolume indicator to measure Pareto front quality. Compare hypervolume across generations to track convergence.

**Expected Output**: Plot of hypervolume vs generation.

<details>
<summary>Hint</summary>
DEAP provides `tools.hypervolume()` function. Need to specify reference point.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

In [ ]:
section_divider("Summary")

## 7. Summary

### Key Takeaways

1. **Multi-Objective Advantage**: Reveals trade-offs instead of hiding them
2. **Pareto Optimality**: Set of non-dominated solutions
3. **NSGA-II**: Efficient multi-objective evolutionary algorithm
4. **Solution Selection**: Knee point, constraints, or preference articulation
5. **Scalability**: Extends to three or more objectives
6. **Flexibility**: User chooses solution based on priorities

### NSGA-II vs Single-Objective GA

| Aspect | Single-Objective | NSGA-II |
|--------|-----------------|----------|
| **Output** | One best solution | Pareto front |
| **Trade-offs** | Hidden in weights | Explicit |
| **User input** | Requires weight tuning | Post-selection |
| **Flexibility** | Fixed weights | Multiple options |
| **Computational cost** | Lower | Higher |

### When to Use Multi-Objective

**Use NSGA-II when:**
- Multiple conflicting objectives exist
- Trade-offs are important to understand
- User preferences are unknown a priori
- Exploration of solution space is desired

**Use single-objective when:**
- One clear objective dominates
- Weights are well-established
- Computational resources are limited
- Single solution needed immediately

### Solution Selection Strategies

1. **Knee point**: Best balance (geometric approach)
2. **Constraints**: Filter by requirements
3. **Reference point**: Closest to ideal
4. **Weighted sum**: Scalarize with preferences
5. **Interactive**: Iterative refinement with user

### What's Next

- **Next notebook**: Hybrid methods (GA + local search)
- **Advanced**: Many-objective optimization (>3 objectives)
- **Applications**: Apply to real multi-criteria problems

### Additional Resources

- **NSGA-II Paper**: Deb et al. (2002) - "A fast and elitist multiobjective genetic algorithm"
- **Multi-Objective Optimization**: Coello Coello et al. (2007) - "Evolutionary Algorithms for Solving Multi-Objective Problems"
- **DEAP Documentation**: https://deap.readthedocs.io/en/master/tutorials/advanced/gp.html

In [ ]:
key_takeaways(["Reveals trade-offs instead of hiding them", "Set of non-dominated solutions", "Efficient multi-objective evolutionary algorithm", "Knee point, constraints, or preference articulation", "Extends to three or more objectives", "User chooses solution based on priorities"])